# Dashboard serving acceptance

Nghiệm thu `lakehouse.sandbox.mart_customer_360_dashboard` sau khi Gold segmentation và masking hoàn thành cho ngày `2026-01-01`. Logic kiểm tra dùng Spark SQL; phần Python chỉ thực thi, assertion và gọi Trino HTTPS.

In [1]:
import base64
import json
import os
import ssl
import urllib.request

try:
    spark
except NameError as exc:
    raise RuntimeError("Hãy chọn kernel 'PySpark (Lakehouse)' trước khi chạy") from exc

COB_DT = "2026-01-01"
results = []

def sql_check(name, query):
    frame = spark.sql(query)
    frame.show(truncate=False)
    status = frame.first()["result"]
    results.append((name, status))
    if status != "PASS":
        raise AssertionError(f"{name}: {status}")

## Grain và population

Dashboard phải có đúng một row/customer/date và khớp hoàn toàn với Gold Customer 360 history của cùng ngày.

In [2]:
sql_check("Dashboard grain and population", f"""
WITH dashboard AS (
    SELECT customer_id, cob_dt
    FROM lakehouse.sandbox.mart_customer_360_dashboard
    WHERE cob_dt = DATE '{COB_DT}'
), stats AS (
    SELECT COUNT(*) AS row_count, COUNT(DISTINCT customer_id) AS unique_customers
    FROM dashboard
), duplicates AS (
    SELECT COUNT(*) AS duplicate_keys
    FROM (SELECT customer_id, cob_dt FROM dashboard GROUP BY customer_id, cob_dt HAVING COUNT(*) > 1)
), mismatch AS (
    SELECT COALESCE(g.customer_id, d.customer_id) AS customer_id
    FROM (SELECT customer_id FROM lakehouse.gold.mart_customer_360_history
          WHERE cob_dt = DATE '{COB_DT}') g
    FULL OUTER JOIN (SELECT customer_id FROM dashboard) d ON g.customer_id = d.customer_id
    WHERE g.customer_id IS NULL OR d.customer_id IS NULL
)
SELECT row_count, unique_customers, duplicate_keys,
       (SELECT COUNT(*) FROM mismatch) AS population_mismatches,
       CASE WHEN row_count = 10000 AND unique_customers = 10000
                  AND duplicate_keys = 0 AND (SELECT COUNT(*) FROM mismatch) = 0
            THEN 'PASS' ELSE 'FAIL' END AS result
FROM stats CROSS JOIN duplicates
""")

+---------+----------------+--------------+---------------------+------+
|row_count|unique_customers|duplicate_keys|population_mismatches|result|
+---------+----------------+--------------+---------------------+------+
|10000    |10000           |0             |0                    |PASS  |
+---------+----------------+--------------+---------------------+------+



## NBO contract

Các trường score, product, reason, priority và suppression phải đầy đủ và nhất quán cho toàn bộ audience.

In [3]:
sql_check("Dashboard NBO contract", f"""
SELECT COUNT(*) AS checked_rows,
       SUM(CASE WHEN cross_sell_score IS NULL OR cross_sell_score NOT BETWEEN 0 AND 100 THEN 1 ELSE 0 END) AS invalid_score,
       SUM(CASE WHEN recommended_product IS NULL OR TRIM(recommended_product) = '' THEN 1 ELSE 0 END) AS missing_product,
       SUM(CASE WHEN recommendation_reason IS NULL OR TRIM(recommendation_reason) = '' THEN 1 ELSE 0 END) AS missing_reason,
       SUM(CASE WHEN campaign_priority IS NULL OR campaign_priority NOT IN ('HIGH','MEDIUM','LOW') THEN 1 ELSE 0 END) AS invalid_priority,
       SUM(CASE WHEN contact_eligible_flag IS NULL OR contact_eligible_flag NOT IN (0,1)
                     OR (contact_eligible_flag = 1 AND suppression_reason IS NOT NULL)
                     OR (contact_eligible_flag = 0 AND suppression_reason IS NULL) THEN 1 ELSE 0 END) AS invalid_suppression,
       CASE WHEN COUNT(*) = 10000
                  AND SUM(CASE WHEN cross_sell_score IS NULL OR cross_sell_score NOT BETWEEN 0 AND 100 THEN 1 ELSE 0 END) = 0
                  AND SUM(CASE WHEN recommended_product IS NULL OR TRIM(recommended_product) = '' THEN 1 ELSE 0 END) = 0
                  AND SUM(CASE WHEN recommendation_reason IS NULL OR TRIM(recommendation_reason) = '' THEN 1 ELSE 0 END) = 0
                  AND SUM(CASE WHEN campaign_priority IS NULL OR campaign_priority NOT IN ('HIGH','MEDIUM','LOW') THEN 1 ELSE 0 END) = 0
                  AND SUM(CASE WHEN contact_eligible_flag IS NULL OR contact_eligible_flag NOT IN (0,1)
                         OR (contact_eligible_flag = 1 AND suppression_reason IS NOT NULL)
                         OR (contact_eligible_flag = 0 AND suppression_reason IS NULL) THEN 1 ELSE 0 END) = 0
            THEN 'PASS' ELSE 'FAIL' END AS result
FROM lakehouse.sandbox.mart_customer_360_dashboard
WHERE cob_dt = DATE '{COB_DT}'
""")

+------------+-------------+---------------+--------------+----------------+-------------------+------+
|checked_rows|invalid_score|missing_product|missing_reason|invalid_priority|invalid_suppression|result|
+------------+-------------+---------------+--------------+----------------+-------------------+------+
|10000       |0            |0              |0             |0               |0                  |PASS  |
+------------+-------------+---------------+--------------+----------------+-------------------+------+



## PII và DA aggregates

Schema không được lộ raw PII. Aggregate bên dưới là dữ liệu đầu vào để đối chiếu dashboard Superset ở nhánh tiếp theo.

In [4]:
raw_pii = {'full_name', 'phone', 'email', 'cccd', 'address'}
dashboard_columns = set(spark.table('lakehouse.sandbox.mart_customer_360_dashboard').columns)
exposed_pii = sorted(raw_pii & dashboard_columns)
pii_status = 'PASS' if not exposed_pii else 'FAIL'
results.append(('Dashboard raw PII absent', pii_status))
print(f'Dashboard schema: {pii_status} ({len(dashboard_columns)} columns, exposed={exposed_pii})')
assert not exposed_pii, f'Raw PII exposed: {exposed_pii}'

spark.sql(f"""
SELECT recommended_product, campaign_priority, contact_eligible_flag,
       COUNT(*) AS customers,
       CAST(SUM(aum_total) AS DECIMAL(24,2)) AS total_aum,
       ROUND(AVG(cross_sell_score), 2) AS avg_cross_sell_score
FROM lakehouse.sandbox.mart_customer_360_dashboard
WHERE cob_dt = DATE '{COB_DT}'
GROUP BY recommended_product, campaign_priority, contact_eligible_flag
ORDER BY customers DESC
""").show(100, truncate=False)

Dashboard schema: PASS (46 columns, exposed=[])


+-------------------+-----------------+---------------------+---------+---------------+--------------------+
|recommended_product|campaign_priority|contact_eligible_flag|customers|total_aum      |avg_cross_sell_score|
+-------------------+-----------------+---------------------+---------+---------------+--------------------+
|CREDIT_CARD        |MEDIUM           |1                    |4613     |216059919000.00|48.17               |
|CREDIT_CARD        |HIGH             |1                    |1838     |777316271000.00|86.72               |
|CREDIT_CARD        |LOW              |0                    |1000     |114712588000.00|27.93               |
|TERM_DEPOSIT       |HIGH             |1                    |722      |888991081000.00|70.0                |
|WEALTH_MANAGEMENT  |HIGH             |1                    |436      |849897068000.00|70.0                |
|DIGITAL_SAVINGS    |MEDIUM           |1                    |282      |39153660000.00 |48.51               |
|TERM_DEPOSIT      

## Trino Marketing RBAC

Marketing phải đọc được dashboard sandbox nhưng vẫn bị từ chối khi query Gold campaign target. Password chỉ được lấy từ environment của container.

In [5]:
TRINO_URL = "https://trino:8443/v1/statement"
TLS_CONTEXT = ssl._create_unverified_context()

def trino_query(user, password, sql):
    auth = base64.b64encode(f"{user}:{password}".encode()).decode()
    request = urllib.request.Request(
        TRINO_URL, data=sql.encode(), method="POST",
        headers={"Authorization": f"Basic {auth}", "X-Trino-User": user},
    )
    payload = json.loads(urllib.request.urlopen(request, context=TLS_CONTEXT).read())
    rows = list(payload.get("data", []))
    while payload.get("nextUri"):
        payload = json.loads(urllib.request.urlopen(payload["nextUri"], context=TLS_CONTEXT).read())
        rows.extend(payload.get("data", []))
    return rows, payload.get("error")

marketing_password = os.environ["TRINO_MARKETING_PASSWORD"]
dashboard_rows, dashboard_error = trino_query(
    "marketing", marketing_password,
    f"SELECT COUNT(*) FROM lakehouse.sandbox.mart_customer_360_dashboard WHERE cob_dt = DATE '{COB_DT}'",
)
_, gold_error = trino_query(
    "marketing", marketing_password,
    "SELECT COUNT(*) FROM lakehouse.gold.campaign_target",
)
security_evidence = [
    ("marketing dashboard allowed", "PASS" if dashboard_rows == [[10000]] and not dashboard_error else "FAIL"),
    ("marketing Gold denied", "PASS" if gold_error and "Access Denied" in gold_error["message"] else "FAIL"),
]
for name, status in security_evidence:
    print(f"{name}: {status}")
    results.append((name, status))
assert all(status == "PASS" for _, status in security_evidence), security_evidence

marketing dashboard allowed: PASS
marketing Gold denied: PASS


In [6]:
assert all(status == 'PASS' for _, status in results), results
print(f"Completed {len(results)}/{len(results)} acceptance checks")
print("DASHBOARD SERVING ACCEPTANCE: PASS")

Completed 5/5 acceptance checks
DASHBOARD SERVING ACCEPTANCE: PASS
